In [ ]:
import gymnasium as gym
import numpy as np

# -------------------------------------------------
# Create FrozenLake Environment
# -------------------------------------------------

env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=True)
env = env.unwrapped

n_states = env.observation_space.n
n_actions = env.action_space.n

gamma = 0.99
theta = 1e-8

# -------------------------------------------------
# Policy Evaluation
# -------------------------------------------------
# Code here
def policy_evaluation(policy, env, gamma=0.99, theta=1e-8):
    """
    Evaluate a policy and return the state-value function.
    """
    n_states = env.observation_space.n
    V = np.zeros(n_states)

    while True:
        delta = 0

        for s in range(n_states):
            v = 0

            for a, action_prob in enumerate(policy[s]):
                for prob, next_state, reward, done in env.P[s][a]:
                    v += action_prob * prob * (
                        reward + gamma * V[next_state] * (not done)
                    )

            delta = max(delta, abs(V[s] - v))
            V[s] = v

        if delta < theta:
            break

    return V

# -------------------------------------------------
# Policy Improvement
# -------------------------------------------------

# Code here
def policy_improvement(V, env, gamma=0.99):
    """
    Improve the policy using the current value function.
    """
    n_states = env.observation_space.n
    n_actions = env.action_space.n

    policy = np.zeros((n_states, n_actions))

    for s in range(n_states):

        action_values = np.zeros(n_actions)

        for a in range(n_actions):
            for prob, next_state, reward, done in env.P[s][a]:
                action_values[a] += prob * (
                    reward + gamma * V[next_state] * (not done)
                )

        best_action = np.argmax(action_values)
        policy[s][best_action] = 1.0

    return policy

# -------------------------------------------------
# Policy Iteration
# -------------------------------------------------

# Code here
def policy_iteration(env, gamma=0.99, theta=1e-8):
    """
    Performs Policy Iteration.
    Returns:
        policy : Optimal Policy
        V      : Optimal State-Value Function
    """

    n_states = env.observation_space.n
    n_actions = env.action_space.n

    # Initialize a random policy
    policy = np.ones((n_states, n_actions)) / n_actions

    iteration = 0  # Count policy iterations

    while True:
        iteration += 1

        # Policy Evaluation
        V = policy_evaluation(policy, env, gamma, theta)

        # Policy Improvement
        new_policy = policy_improvement(V, env, gamma)

        # Check if policy has converged
        if np.array_equal(np.argmax(policy, axis=1),
                          np.argmax(new_policy, axis=1)):
            print("Policy converged!")
            print(f"Total policy iterations: {iteration}")
            break

        policy = new_policy

    return new_policy, V


# -------------------------------------------------
# Display Functions
# -------------------------------------------------

def print_value_function(V):
    print("\nOptimal State-Value Function:")
    print(np.round(V.reshape(4, 4), 4))


def print_policy(policy):
    action_symbols = {
        0: "←",
        1: "↓",
        2: "→",
        3: "↑"
    }

    best_actions = np.argmax(policy, axis=1)
    policy_grid = np.array(
        [action_symbols[action] for action in best_actions]
    ).reshape(4, 4)


    print("\nOptimal Policy:")
    print(policy_grid)

# -------------------------------------------------
# Run Policy Iteration
# -------------------------------------------------

optimal_policy, optimal_value_function = policy_iteration(
    env,
    gamma=gamma,
    theta=theta
)

print("Name: VENKATANATHAN P R")
print("Register Number: 212223240173")
print_value_function(optimal_value_function)
print_policy(optimal_policy)

env.close()


Policy converged!
Total policy iterations: 3
Name: VENKATANATHAN P R
Register Number: 212223240173

Optimal State-Value Function:
[[0.542  0.4988 0.4707 0.4569]
 [0.5585 0.     0.3583 0.    ]
 [0.5918 0.6431 0.6152 0.    ]
 [0.     0.7417 0.8628 0.    ]]

Optimal Policy:
[['←' '↑' '↑' '↑']
 ['←' '←' '←' '←']
 ['↑' '↓' '←' '←']
 ['←' '→' '↓' '←']]
